In [ ]:
# === GOOGLE COLAB SETUP ===
import os
if 'COLAB_GPU' in os.environ:
    print("Running on Google Colab. Setting up environment...")
    # Fixed pathing: Ensure we are in /content/btc_ai exactly
    os.chdir('/content')
    if not os.path.exists('btc_ai'):
        !git clone https://github.com/taybro-o/btc_ai.git
    os.chdir('/content/btc_ai')
    !git pull origin master  # Get the latest code/data updates
    !pip install -q scikit-learn matplotlib pandas numpy tensorflow requests
    print(f"Current Working Directory: {os.getcwd()}")
    print("Environment ready!")
else:
    print("Running locally. Skipping Colab setup.")

# BTC AI Master Notebook (PatchTST Optimization)
This notebook handles data updates, feature engineering, and training a PatchTST model for 5-minute price forecasting using live Binance data.

## 1. Data Acquisition (Live Binance API)
Fetches the latest 1-minute candles directly from Binance mirrors.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import requests
from tensorflow.keras import layers
from datetime import datetime, timedelta

DATA_DIR = "data"
MASTER_FILE = os.path.join(DATA_DIR, "btcusdt_analysis_data.csv")

def get_binance_data(symbol="BTCUSDT", interval="1m", limit=1000):
    """Fetches the latest candles from multiple Binance mirrors (US fallback included)."""
    mirrors = [
        "https://api.binance.us",   # Often works on US-based Colab servers
        "https://api.binance.com",
        "https://api1.binance.com",
        "https://api2.binance.com",
        "https://api3.binance.com"
    ]
    
    for base_url in mirrors:
        url = f"{base_url}/api/v3/klines?symbol={symbol}&interval={interval}&limit={limit}"
        try:
            print(f"Trying Binance API: {base_url}...")
            response = requests.get(url, timeout=10)
            if response.status_code == 200:
                data = response.json()
                df = pd.DataFrame(data, columns=[
                    'Timestamp', 'Open', 'High', 'Low', 'Close', 'Volume', 
                    'Close_time', 'Quote_av', 'Trades', 'Tb_base_av', 'Tb_quote_av', 'Ignore'
                ])
                cols = ['Open', 'High', 'Low', 'Close', 'Volume']
                df[cols] = df[cols].apply(pd.to_numeric)
                df['Timestamp'] = df['Timestamp'] // 1000
                return df[['Timestamp', 'Open', 'High', 'Low', 'Close', 'Volume']]
            else:
                print(f"Mirror {base_url} returned status: {response.status_code}")
        except Exception as e:
            print(f"Mirror {base_url} failed: {e}")
    
    return None

def update_and_prune_data():
    print("Fetching live data from Binance...")
    new_df = get_binance_data()
    
    if os.path.exists(MASTER_FILE):
        print(f"Found existing data file at {MASTER_FILE}")
        master_df = pd.read_csv(MASTER_FILE)
        if new_df is not None and not new_df.empty:
            df = pd.concat([master_df, new_df]).drop_duplicates(subset=['Timestamp'], keep='last')
        else:
            print("API failed to fetch new data (Region Blocked). Using cached data from repo.")
            df = master_df
    else:
        if new_df is not None and not new_df.empty:
            print("No existing file found. Starting fresh with API data.")
            df = new_df
        else:
            # Total fallback if API fails and file is missing (should not happen after fix)
            raise ValueError(f"CRITICAL: No API data and no file found at {os.path.abspath(MASTER_FILE)}.")

    # Clean NaN and keep last 10k rows
    df = df.dropna(subset=['Timestamp', 'Close'])
    df = df.sort_values('Timestamp').tail(10000)
    
    if not os.path.exists(DATA_DIR): os.makedirs(DATA_DIR)
    df.to_csv(MASTER_FILE, index=False)
    
    max_ts = df['Timestamp'].max()
    last_ts = datetime.fromtimestamp(max_ts)
    print(f"Data updated. Rows: {len(df)}. Latest candle: {last_ts} UTC")
    return df

df = update_and_prune_data()
df['Datetime'] = pd.to_datetime(df['Timestamp'], unit='s')
df.set_index('Datetime', inplace=True)
df.sort_index(inplace=True)

## 2. Feature Engineering
Adding technical indicators using vectorized operations.

In [ ]:
def apply_indicators(df):
    # Log Returns
    df['log_return'] = np.log(df['Close'] / df['Close'].shift(1))
    
    # EMA, RSI, MACD, ADX
    df['EMA_50'] = df['Close'].ewm(span=50).mean()
    delta = df['Close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(14).mean()
    df['RSI'] = 100 - (100 / (1 + (gain / (loss + 1e-9))))
    df['MACD'] = df['Close'].ewm(span=12).mean() - df['Close'].ewm(span=26).mean()
    
    # ADX
    tr = pd.concat([df['High']-df['Low'], abs(df['High']-df['Close'].shift(1)), abs(df['Low']-df['Close'].shift(1))], axis=1).max(axis=1)
    atr = tr.rolling(14).mean()
    plus_di = 100 * ((df['High'].diff()).clip(lower=0).rolling(14).mean() / (atr + 1e-9))
    minus_di = 100 * ((df['Low'].diff() * -1).clip(lower=0).rolling(14).mean() / (atr + 1e-9))
    df['ADX'] = 100 * (abs(plus_di - minus_di) / (plus_di + minus_di + 1e-9)).rolling(14).mean()
    
    df.dropna(inplace=True)
    return df

df = apply_indicators(df)
print(f"Features created. Shape: {df.shape}")

## 3. Data Preparation for PatchTST
Scaling data and creating overlapping sequences.

In [ ]:
from sklearn.preprocessing import StandardScaler

LOOKBACK = 96  # 96 minutes input
FORECAST = 5   # 5 minutes prediction

features = ['Open', 'High', 'Low', 'Close', 'Volume', 'EMA_50', 'RSI', 'MACD', 'ADX', 'log_return']
scaler = StandardScaler()
scaled_data = scaler.fit_transform(df[features])

def create_sequences(data, lookback, forecast):
    X, y = [], []
    for i in range(len(data) - lookback - forecast + 1):
        X.append(data[i : i + lookback])
        # Predict the next 5 LOG RETURNS (last column)
        y.append(data[i + lookback : i + lookback + forecast, -1]) 
    return np.array(X), np.array(y)

X, y = create_sequences(scaled_data, LOOKBACK, FORECAST)

# Split: 90% Train, 10% Val
split = int(len(X) * 0.9)
X_train, X_val = X[:split], X[split:]
y_train, y_val = y[:split], y[split:]
print(f"X_train shape: {X_train.shape}")

## 4. PatchTST Model Architecture (TensorFlow)
A custom Keras implementation of the PatchTST model.

In [ ]:
class PatchTST(tf.keras.Model):
    def __init__(self, num_patches, patch_len, model_dim, num_heads, num_layers, forecast_len):
        super().__init__()
        self.patch_len = patch_len
        self.num_patches = num_patches
        self.model_dim = model_dim
        
        self.linear_proj = layers.Dense(model_dim)
        self.dropout = layers.Dropout(0.1)
        
        self.transformer_blocks = [
            layers.MultiHeadAttention(num_heads=num_heads, key_dim=model_dim//num_heads)
            for _ in range(num_layers)
        ]
        self.layer_norms = [layers.LayerNormalization() for _ in range(num_layers * 2)]
        
        self.flatten = layers.Flatten()
        self.head = layers.Dense(forecast_len)

    def build(self, input_shape):
        self.pos_encoding = self.add_weight(
            name="pos_enc", 
            shape=(1, self.num_patches, self.model_dim), 
            initializer="random_normal", 
            trainable=True
        )
        super().build(input_shape)

    def call(self, x):
        batch_size = tf.shape(x)[0]
        feat_dim = tf.shape(x)[-1]
        
        x = tf.reshape(x, (batch_size, self.num_patches, self.patch_len * feat_dim))
        x = self.linear_proj(x)
        x = x + self.pos_encoding
        
        for i, block in enumerate(self.transformer_blocks):
            attn_out = block(x, x)
            x = self.layer_norms[i*2](x + attn_out)
            x = self.dropout(x)
            x = self.layer_norms[i*2 + 1](x)
            
        x = self.flatten(x)
        return self.head(x)

# Hyperparameters
PATCH_LEN = 16
NUM_PATCHES = LOOKBACK // PATCH_LEN
MODEL_DIM = 64
NUM_HEADS = 4
NUM_LAYERS = 2

def directional_accuracy(y_true, y_pred):
    true_sign = tf.sign(y_true)
    pred_sign = tf.sign(y_pred)
    return tf.reduce_mean(tf.cast(tf.equal(true_sign, pred_sign), tf.float32))

model = PatchTST(NUM_PATCHES, PATCH_LEN, MODEL_DIM, NUM_HEADS, NUM_LAYERS, FORECAST)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4), 
    loss='mse',
    metrics=['mae', directional_accuracy]
)

sample_x = X_train[:1]
_ = model(sample_x)
model.summary()

## 5. Training
Estimated time: ~2-5 mins on T4 GPU.

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

callbacks = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1)
]

history = model.fit(
    X_train, y_train, 
    validation_data=(X_val, y_val), 
    epochs=30, 
    batch_size=256, 
    callbacks=callbacks,
    verbose=1
)

plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.legend()
plt.show()

## 6. Prediction & Forecast Output

In [ ]:
predictions = model.predict(X_val)
return_mean = scaler.mean_[-1]
return_std = scaler.scale_[-1]

def denormalize_returns(data):
    return (data * return_std) + return_mean

pred_returns = denormalize_returns(predictions)
last_price = df['Close'].iloc[-1]

print("\n--- 5-MINUTE FORECAST ---")
for i in range(5):
    forecasted_return = pred_returns[-1, i]
    direction = "UP" if forecasted_return > 0 else "DOWN"
    projected_price = last_price * np.exp(forecasted_return)
    print(f"Minute {i+1}: {direction} ({projected_price:.2f} USD)")
    last_price = projected_price